# Stage 8 — Convert to GGUF

**Goal:** convert the training-format model into an inference-format model, and
**prove the conversion was lossless**.

### Why a second format exists at all

| | safetensors (training) | GGUF (inference) |
|---|---|---|
| Needs | PyTorch, Python, CUDA | one C++ binary |
| Layout | tensor blobs + separate JSON config | tensors *and* all metadata in one file |
| Weights | fp32/fp16 | fp16 or quantized (Q8_0, Q4_K_M, …) |
| Optimised for | gradients, flexibility | memory bandwidth, mmap, fast startup |

GGUF is self-contained: architecture, hyperparameters, the tokenizer vocabulary,
*and the chat template* all live inside the single file. That is what lets
`llama-server -m model.gguf` work with no config, no Python, and no tokenizer
files sitting next to it.

### This is where stage 2's decision pays off

`convert_hf_to_gguf.py` tries `_set_vocab_sentencepiece()` **first**, which looks
for a file named exactly `tokenizer.model`. We have one, so conversion takes that
path and never reaches `_set_vocab_gpt2()` — the one that hashes the pre-tokenizer
against a hardcoded registry and raises `NotImplementedError` for custom
vocabularies.

**Conversion runs here, on Colab, deliberately.** It needs PyTorch to read the
safetensors, and the local Windows tier has no PyTorch by design — that's what
keeps the local footprint at ~700 MB.

In [ ]:
# --- Colab bootstrap -------------------------------------------------------
# Set this to YOUR GitHub repo once; every notebook uses the same cell.
REPO_URL = "https://github.com/pythonstudentiam/e2e_llm_demo.git"

import os, subprocess, sys
from pathlib import Path

REPO = Path("/content/e2e_llm_demo")
WORK = Path("/content/work")          # scratch: data + checkpoints (ephemeral!)
WORK.mkdir(parents=True, exist_ok=True)

if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)

sys.path.insert(0, str(REPO / "src"))

# Colab ships torch; these are the rest. -q to keep the log readable.
%pip install -q sentencepiece "datasets>=3.0" "transformers>=4.45" "huggingface_hub>=0.30"

# HF token from the Colab Secrets panel (key icon, left sidebar). Name it
# HF_TOKEN and enable Notebook access -- the grant is PER NOTEBOOK, so every
# notebook asks separately. Never paste a token into a cell.
#
# login() rather than just setting the env var: it writes the token where every
# huggingface_hub call looks, including ones that ignore the environment.
try:
    from google.colab import userdata
    from huggingface_hub import login

    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
    print("HF_TOKEN loaded from Colab Secrets, authenticated")
except Exception as e:
    print("=" * 72)
    print(f"  HF_TOKEN IS NOT AVAILABLE  ({type(e).__name__}: {e})")
    print()
    print("  Every Hub call in this notebook will fail with 401 Unauthorized.")
    print("  Fix: click the key icon in the left sidebar, turn on Notebook")
    print("       access for HF_TOKEN, then RE-RUN THIS CELL before continuing.")
    print("=" * 72)

import torch
print(f"torch {torch.__version__} | CUDA {torch.cuda.is_available()} | "
      f"{torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")

In [ ]:
from tinyllm import config
from tinyllm.config import (
    model_cfg, train_cfg, data_cfg, tok_cfg, sft_cfg, gen_cfg, quant_cfg, serve_cfg, hub,
)

print(config.summary())

## 8.1 — Get llama.cpp

Only the Python conversion script is needed, so this is a shallow clone with no
C++ build.

In [ ]:
import subprocess
from pathlib import Path

LLAMA = Path("/content/llama.cpp")
if not LLAMA.exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/ggml-org/llama.cpp", str(LLAMA)], check=True)

# Deliberately NOT installing llama.cpp's requirements file. It pins its own
# huggingface_hub and transformers, which pip then installs over the ones this
# session has already imported -- leaving half-updated modules and errors like
# "module 'huggingface_hub.constants' has no attribute HF_HUB_ENABLE_HF_TRANSFER".
#
# It isn't needed: convert_hf_to_gguf.py puts its own bundled gguf-py on
# sys.path, and Colab already has torch, transformers and safetensors.
%pip install -q sentencepiece protobuf

rev = subprocess.run(["git", "-C", str(LLAMA), "rev-parse", "--short", "HEAD"],
                     capture_output=True, text=True).stdout.strip()
print(f"llama.cpp @ {rev}")

## 8.2 — Fetch the model we published

Downloading from the Hub rather than reusing the local export is the point: it
proves the *published* artifact converts, which is the one anyone else would get.

In [ ]:
from huggingface_hub import snapshot_download

model_dir = Path(snapshot_download(repo_id=hub.model_repo, local_dir="/content/hf_model"))
print("downloaded:")
for f in sorted(model_dir.iterdir()):
    if f.is_file():
        print(f"  {f.name:<30} {f.stat().st_size:>10,} B")

assert (model_dir / "tokenizer.model").exists(), (
    "tokenizer.model is missing -- conversion would fall through to the hashed "
    "BPE path and fail. Re-run notebook 07."
)
print("\ntokenizer.model present -> SentencePiece path confirmed")

## 8.3 — Convert

In [ ]:
gguf_dir = WORK / "gguf"
gguf_dir.mkdir(parents=True, exist_ok=True)
f16_path = gguf_dir / f"{config.PROJECT_NAME}-f16.gguf"

result = subprocess.run(
    [sys.executable, str(LLAMA / "convert_hf_to_gguf.py"), str(model_dir),
     "--outfile", str(f16_path), "--outtype", "f16"],
    capture_output=True, text=True,
)
print(result.stdout[-3000:])
if result.returncode != 0:
    print("STDERR:\n", result.stderr[-3000:])
    raise RuntimeError("conversion failed")

print(f"\n{f16_path.name}: {f16_path.stat().st_size / 1e6:.1f} MB")

## 8.4 — Inspect the metadata

Everything the runtime needs is now inside the file. Worth reading once — this is
what "self-contained" actually means.

In [ ]:
import numpy as np

sys.path.insert(0, str(LLAMA / "gguf-py"))
from gguf import GGUFReader

reader = GGUFReader(str(f16_path))

print("KEY METADATA")
interesting = ("general.architecture", "general.name", "llama.block_count",
               "llama.embedding_length", "llama.attention.head_count",
               "llama.attention.head_count_kv", "llama.context_length",
               "llama.feed_forward_length", "llama.rope.freq_base",
               "tokenizer.ggml.model", "tokenizer.ggml.bos_token_id",
               "tokenizer.ggml.eos_token_id")
for field in reader.fields.values():
    if field.name in interesting:
        try:
            val = field.contents()
        except Exception:
            val = "<complex>"
        print(f"  {field.name:<40} {val}")

print(f"\n  tensors: {len(reader.tensors)}")
total = sum(int(np.prod(t.shape)) for t in reader.tensors)
print(f"  total parameters in file: {total:,}")
print(f"  config.py says:           {model_cfg.n_params:,}")

In [ ]:
# The chat template travels inside the GGUF -- this is hop 3 of 4 for that string.
for field in reader.fields.values():
    if "chat_template" in field.name:
        print(f"{field.name}:\n")
        print(field.contents()[:600])
        break
else:
    print("WARNING: no chat template in the GGUF. llama-server will fall back to a")
    print("generic format and the model will see prompts it was not trained on.")

## 8.5 — The stage 8 gate: prove conversion was lossless

Run the *same* prompt through `transformers` and through `llama.cpp` with greedy
decoding (temperature 0, so there's no sampling randomness to hide behind). The
token sequences must match.

This is a genuine end-to-end check across two entirely separate implementations —
one Python, one C++. If a tensor were transposed, a rope parameter misread, or the
vocabulary misordered, the outputs would diverge.

In [ ]:
# Build just the CLI. ~3-4 minutes.
subprocess.run(["cmake", "-B", "build", "-DGGML_NATIVE=OFF", "-DLLAMA_CURL=OFF"],
               cwd=LLAMA, check=True, capture_output=True)
r = subprocess.run(["cmake", "--build", "build", "--target", "llama-cli",
                    "-j", str(os.cpu_count())], cwd=LLAMA, capture_output=True, text=True)
if r.returncode != 0:
    print(r.stdout[-2000:]); print(r.stderr[-2000:])
    raise RuntimeError("llama.cpp build failed")

cli = LLAMA / "build/bin/llama-cli"
print(f"built {cli}")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

PROMPT = "Once upon a time, there was a little girl named Lily."
N = 40

# --- reference: transformers, greedy
tok = AutoTokenizer.from_pretrained(str(model_dir))
hf_model = AutoModelForCausalLM.from_pretrained(str(model_dir)).eval()
ids = tok(PROMPT, return_tensors="pt")
with torch.no_grad():
    out = hf_model.generate(**ids, max_new_tokens=N, do_sample=False,
                            num_beams=1, temperature=None, top_p=None, top_k=None)
hf_text = tok.decode(out[0], skip_special_tokens=True)
print("TRANSFORMERS (greedy):")
print(f"  {hf_text}\n")

In [ ]:
# --- llama.cpp, greedy (temp 0), same prompt, same token budget
r = subprocess.run(
    [str(cli), "-m", str(f16_path), "-p", PROMPT, "-n", str(N),
     "--temp", "0", "--seed", "0", "-no-cnv", "--no-warmup"],
    capture_output=True, text=True,
)
cpp_text = r.stdout.strip()
print("LLAMA.CPP (greedy):")
print(f"  {cpp_text}\n")

def norm(s):
    return " ".join(s.replace(PROMPT, "", 1).split())

a, b = norm(hf_text), norm(cpp_text)
print("=" * 78)
if a == b:
    print("MATCH -- the conversion is lossless.")
else:
    # Compare word by word to locate the first divergence.
    wa, wb = a.split(), b.split()
    n_same = next((i for i, (x, y) in enumerate(zip(wa, wb)) if x != y), min(len(wa), len(wb)))
    print(f"First {n_same} words agree, then diverge.")
    print(f"  transformers: ...{' '.join(wa[max(0,n_same-3):n_same+5])}")
    print(f"  llama.cpp:    ...{' '.join(wb[max(0,n_same-3):n_same+5])}")
    print("\nA few words of agreement then drift is usually fp16 rounding, which is")
    print("benign. Divergence from the very first token means a real conversion bug.")

## 8.6 — Publish the GGUF

Both formats in one repo is the convention: safetensors for anyone who wants to
fine-tune, GGUF for anyone who wants to run it.

In [ ]:
from tinyllm.export import upload_gguf

print(upload_gguf(f16_path))

## Stage 8 gate

- [x] Converted via the SentencePiece path — no pre-tokenizer hash error
- [x] Metadata verified: architecture, GQA head counts, rope params, vocab
- [x] Chat template present inside the GGUF
- [x] Greedy output matches `transformers` — conversion is lossless
- [x] f16 GGUF published to the Hub

---

## The GPU tier is done

Everything from here runs on your laptop. Head back to the terminal:

```powershell
.\scripts\pull_model.ps1     # fetch the f16 GGUF from your Hub repo
.\scripts\quantize.ps1       # Q8_0, Q5_K_M, Q4_K_M
.\scripts\serve.ps1          # llama-server on localhost:8080
```

Then open `notebooks/local/09_inspect_gguf.ipynb`.